In [ ]:
'''
taco_full_dataset.parquet을 받아서 두 개의 csv(problems.csv, problem_testcases.csv)로 분리하는 코드를 작성하고 싶어. 참고로 parquet의 용량이 크니 1000개 정도의 배치사이즈로 처리하도록 코드를 작성해줘. 미결정된 부분은 우선 질문부터 하고.

<problems.csv>
- parquet에서 input_output을 제외한 모든 열을 다 똑같이 가져오되, id라는 열을 가장 왼쪽에 새로 만들어서 첫 번째 행을 1, 그 다음부터 1씩 증가하도록 값을 부여한다.
- 열의 순서를 다음처럼 재배치한다 : id, question, count_cases, difficulty, raw_tags, tags, skill_types, time_limit, memory_limit, Expected Time Complexity, Expected Auxiliary Space, (나머지 열들은 오른쪽에, 순서 상관없음)

<problem_testcases.csv>
- 열 이름 : problem_id, testcase_order, input, output, validation
- problem_id : 해당 테스트 케이스의 문제번호(problems.csv의 id값)
- testcase_order : 동일 problem_id 내에서의 순서 (위에서부터 1, 2, 3, ...)
- input & output : 원본 parquet에서 input_output = {"inputs":[입력1, 입력2, ...], "outputs":[출력1, 출력2, ...]} 에서 각각 입력n, 출력n을 가져온 값
- validation : 현재는 빈 값(true/false 둘 다 아닌, null)
'''

In [1]:
import pandas as pd
import pyarrow.parquet as pq
import json
import ast
import sys
import os

# 이전 에러 방지: 파이썬 정수-문자열 변환 최대 자릿수 제한 해제
sys.set_int_max_str_digits(100000)

def parse_input_output(io_data):
    """
    다양한 형태(str, dict)의 input_output 데이터를 안전하게 딕셔너리로 파싱하는 함수
    """
    if pd.isna(io_data):
        return {"inputs": [], "outputs": []}
    
    if isinstance(io_data, dict):
        return io_data
        
    try:
        return json.loads(io_data)
    except:
        try:
            return ast.literal_eval(io_data)
        except:
            return {"inputs": [], "outputs": []}

def parse_solutions(sol_data):
    """
    문자열 형태의 solutions 데이터를 안전하게 리스트로 파싱하는 함수
    """
    if pd.isna(sol_data):
        return []
    
    if isinstance(sol_data, list):
        return sol_data
        
    try:
        return json.loads(sol_data)
    except:
        try:
            return ast.literal_eval(sol_data)
        except:
            return []

def split_taco_dataset(input_parquet_path, prob_csv_path, tc_csv_path, sol_csv_path, batch_size=1000):
    print(f"🚀 작업을 시작합니다. (Batch Size: {batch_size})")
    
    # 기존 파일이 있다면 덮어쓰기 위해 삭제 (Append 모드 충돌 방지)
    if os.path.exists(prob_csv_path): os.remove(prob_csv_path)
    if os.path.exists(tc_csv_path): os.remove(tc_csv_path)
    if os.path.exists(sol_csv_path): os.remove(sol_csv_path)

    parquet_file = pq.ParquetFile(input_parquet_path)
    global_id_counter = 1
    batch_count = 1
    
    is_first_batch = True # 첫 배치에만 CSV 헤더(컬럼명)를 쓰기 위한 플래그

    # 지정된 사이즈만큼 청크 단위로 읽어오기
    for batch in parquet_file.iter_batches(batch_size=batch_size):
        df_batch = batch.to_pandas()
        current_batch_size = len(df_batch)
        
        print(f"⏳ 처리 중... [배치 {batch_count}] (ID: {global_id_counter} ~ {global_id_counter + current_batch_size - 1})")
        
        # ---------------------------------------------------------
        # 1. problems.csv 데이터 구성
        # ---------------------------------------------------------
        df_problems = df_batch.copy()
        
        # id 부여
        df_problems['id'] = range(global_id_counter, global_id_counter + current_batch_size)
        
        # count_cases 열 추가 (inputs의 개수)
        df_problems['count_cases'] = df_problems['input_output'].apply(
            lambda x: len(parse_input_output(x).get('inputs', []))
        )
        
        # count_solutions 열 추가 및 기존 solutions 열 삭제
        if 'solutions' in df_problems.columns:
            df_problems['count_solutions'] = df_problems['solutions'].apply(
                lambda x: len(parse_solutions(x))
            )
            df_problems = df_problems.drop(columns=['solutions'])
        else:
            df_problems['count_solutions'] = 0
        
        # input_output 열 삭제
        if 'input_output' in df_problems.columns:
            df_problems = df_problems.drop(columns=['input_output'])
            
        # 열 순서 재배치 (count_cases 옆에 count_solutions 추가)
        desired_front_cols = [
            'id', 'question', 'count_cases', 'count_solutions', 'difficulty', 'raw_tags', 'tags', 'skill_types', 
            'time_limit', 'memory_limit', 'Expected Time Complexity', 'Expected Auxiliary Space'
        ]
        
        actual_columns = df_problems.columns.tolist()
        # 실제 존재하는 앞부분 열만 필터링 (없는 열은 무시)
        front_cols = [col for col in desired_front_cols if col in actual_columns]
        # 나머지 열들
        back_cols = [col for col in actual_columns if col not in front_cols]
        
        df_problems = df_problems[front_cols + back_cols]
        
        # ---------------------------------------------------------
        # 2. testcases 및 solutions 데이터 구성
        # ---------------------------------------------------------
        testcases_list = []
        solutions_list = []
        
        for idx, row in df_batch.iterrows():
            current_prob_id = global_id_counter + idx # 현재 행의 id
            
            # [테스트 케이스 처리]
            io_data = row.get('input_output', None)
            parsed_io = parse_input_output(io_data)
            
            inputs = parsed_io.get('inputs', [])
            outputs = parsed_io.get('outputs', [])
            
            for tc_order, (inp, out) in enumerate(zip(inputs, outputs), start=1):
                testcases_list.append({
                    'problem_id': current_prob_id,
                    'testcase_order': tc_order,
                    'input': str(inp),
                    'output': str(out),
                    'validation': None
                })
                
            # [솔루션 처리]
            sol_data = row.get('solutions', None)
            parsed_sols = parse_solutions(sol_data)
            
            for sol_order, sol in enumerate(parsed_sols, start=1):
                solutions_list.append({
                    'problem_id': current_prob_id,
                    'solution_order': sol_order,
                    'solution': str(sol),  # 문자열 코드 저장
                    'validation': None     # 빈 값 처리
                })
                
        df_testcases = pd.DataFrame(testcases_list)
        df_solutions = pd.DataFrame(solutions_list)
        
        # ---------------------------------------------------------
        # 3. CSV로 저장 (Append 모드)
        # ---------------------------------------------------------
        write_mode = 'w' if is_first_batch else 'a'
        write_header = is_first_batch
        
        # index=False 로 인덱스 열 제거
        df_problems.to_csv(prob_csv_path, mode=write_mode, index=False, header=write_header, encoding='utf-8-sig')
        # 데이터가 없을 경우(빈 데이터프레임) 저장 시 에러 방지
        if not df_testcases.empty:
            df_testcases.to_csv(tc_csv_path, mode=write_mode, index=False, header=write_header, encoding='utf-8-sig')
        if not df_solutions.empty:
            df_solutions.to_csv(sol_csv_path, mode=write_mode, index=False, header=write_header, encoding='utf-8-sig')
        
        # 다음 배치를 위한 업데이트
        global_id_counter += current_batch_size
        is_first_batch = False
        batch_count += 1

    print(f"\n✅ 분리 완료! 총 {global_id_counter - 1}개의 문제가 성공적으로 분리되었습니다.")
    print(f"📁 문제 파일: {prob_csv_path}")
    print(f"📁 테스트케이스 파일: {tc_csv_path}")
    print(f"📁 솔루션 파일: {sol_csv_path}")

# ==========================================
# 실행 부분
# ==========================================
# 파일 경로 설정 (환경에 맞게 수정하세요)
INPUT_PARQUET = "taco_full_dataset.parquet"
OUTPUT_PROBLEMS_CSV = "problems.csv"
OUTPUT_TESTCASES_CSV = "problem_testcases.csv"
OUTPUT_SOLUTIONS_CSV = "problem_solutions.csv"

# 함수 실행 (원하는 경우 batch_size 조절 가능)
split_taco_dataset(INPUT_PARQUET, OUTPUT_PROBLEMS_CSV, OUTPUT_TESTCASES_CSV, OUTPUT_SOLUTIONS_CSV, batch_size=1000)

🚀 작업을 시작합니다. (Batch Size: 1000)
⏳ 처리 중... [배치 1] (ID: 1 ~ 1000)
⏳ 처리 중... [배치 2] (ID: 1001 ~ 2000)
⏳ 처리 중... [배치 3] (ID: 2001 ~ 3000)
⏳ 처리 중... [배치 4] (ID: 3001 ~ 4000)
⏳ 처리 중... [배치 5] (ID: 4001 ~ 5000)
⏳ 처리 중... [배치 6] (ID: 5001 ~ 6000)
⏳ 처리 중... [배치 7] (ID: 6001 ~ 7000)
⏳ 처리 중... [배치 8] (ID: 7001 ~ 8000)
⏳ 처리 중... [배치 9] (ID: 8001 ~ 9000)
⏳ 처리 중... [배치 10] (ID: 9001 ~ 10000)
⏳ 처리 중... [배치 11] (ID: 10001 ~ 11000)
⏳ 처리 중... [배치 12] (ID: 11001 ~ 12000)
⏳ 처리 중... [배치 13] (ID: 12001 ~ 13000)
⏳ 처리 중... [배치 14] (ID: 13001 ~ 14000)
⏳ 처리 중... [배치 15] (ID: 14001 ~ 15000)
⏳ 처리 중... [배치 16] (ID: 15001 ~ 16000)
⏳ 처리 중... [배치 17] (ID: 16001 ~ 17000)
⏳ 처리 중... [배치 18] (ID: 17001 ~ 18000)
⏳ 처리 중... [배치 19] (ID: 18001 ~ 19000)
⏳ 처리 중... [배치 20] (ID: 19001 ~ 20000)
⏳ 처리 중... [배치 21] (ID: 20001 ~ 21000)
⏳ 처리 중... [배치 22] (ID: 21001 ~ 22000)
⏳ 처리 중... [배치 23] (ID: 22001 ~ 23000)
⏳ 처리 중... [배치 24] (ID: 23001 ~ 24000)
⏳ 처리 중... [배치 25] (ID: 24001 ~ 25000)
⏳ 처리 중... [배치 26] (ID: 25001 ~ 25443)

✅ 분리 완료! 총

In [ ]:
'''
세 개의 csv를 각각 테이블 display로 상위 5개 행씩 보여주는 코드를 작성해줘.
'''

In [1]:
import pandas as pd
from IPython.display import display

# 파일 경로 (앞서 저장한 파일명과 동일해야 합니다)
PROB_CSV_PATH = "problems.csv"
TC_CSV_PATH = "problem_testcases.csv"
SL_CSV_PATH = "problem_solutions.csv"

def display_csv_samples():
    try:
        # 1. problems.csv 확인
        print("📊 [problems.csv] 데이터 상위 5개 행")
        # 전체 파일을 다 읽지 않고, nrows=5를 통해 딱 5줄만 메모리에 올립니다.
        df_problems = pd.read_csv(PROB_CSV_PATH, nrows=5)
        display(df_problems)
        
        print("-" * 80)
        
        # 2. problem_testcases.csv 확인
        print("\n📊 [problem_testcases.csv] 데이터 상위 5개 행")
        df_testcases = pd.read_csv(TC_CSV_PATH, nrows=5)
        display(df_testcases)
        
        # 3. problem_solutions.csv 확인
        print("\n📊 [problem_solutions.csv] 데이터 상위 5개 행")
        df_solutions = pd.read_csv(SL_CSV_PATH, nrows=5)
        display(df_solutions)
    except FileNotFoundError as e:
        print(f"❌ 파일을 찾을 수 없습니다. 경로를 확인해 주세요.\n{e}")
    except Exception as e:
        print(f"❌ 데이터를 불러오는 중 에러가 발생했습니다:\n{e}")

# 확인 함수 실행
display_csv_samples()

📊 [problems.csv] 데이터 상위 5개 행


,id,question,count_cases,count_solutions,difficulty,raw_tags,tags,skill_types,time_limit,memory_limit,Expected Time Complexity,Expected Auxiliary Space,starter_code,name,source,url,date,picture_num
0,1,This is an interactive problem.\n\nIn good old...,134,0,HARD,"['interactive', 'binary search', 'geometry', '...",['Geometry' 'Sorting' 'Constructive algorithms'],['Sorting'],2.0 seconds,256.0 megabytes,NaN,NaN,NaN,NaN,codeforces,https://codeforces.com/problemset/problem/1063/C,NaN,NaN
1,2,There are $n$ candy boxes in front of Tania. T...,94,10,HARD,['dp'],['Dynamic programming'],['Dynamic programming'],NaN,NaN,NaN,NaN,NaN,NaN,codeforces,https://codeforces.com/problemset/problem/1057/C,2019-12-31,NaN
2,3,Little Petya likes to play a lot. Most of all ...,54,0,VERY_HARD,"['data structures', 'dsu']",['Spanning trees' 'Data structures'],['Data structures'],1.0 seconds,64.0 megabytes,NaN,NaN,NaN,NaN,codeforces,https://codeforces.com/problemset/problem/13/E,NaN,NaN
3,4,"If you visit Aizu Akabeko shrine, you will fin...",103,2,UNKNOWN_DIFFICULTY,[],[],[],1.0 seconds,268.435456 megabytes,NaN,NaN,NaN,NaN,aizu,NaN,NaN,NaN
4,5,"You have a deck of $n$ cards, and you'd like t...",4,170,EASY,"['data structures', 'greedy', 'math']",['Data structures' 'Mathematics' 'Greedy algor...,"['Data structures', 'Greedy algorithms']",1 second,512 megabytes,NaN,NaN,NaN,NaN,codeforces,https://codeforces.com/problemset/problem/1492/B,2021-02-23,0.0


--------------------------------------------------------------------------------

📊 [problem_testcases.csv] 데이터 상위 5개 행


,problem_id,testcase_order,input,output,validation
0,1,1,hack\n30\n1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 ...,0 1\n0 0 1000000000 2\n,NaN
1,1,2,random\n22\n2\n,0 1\n0 0 1000000000 2\n,NaN
2,1,3,random\n20\n11\n,0 1\n0 0 1000000000 2\n,NaN
3,1,4,random\n10\n1\n,0 1\n0 0 1000000000 2\n,NaN
4,1,5,random\n20\n12\n,0 1\n0 0 1000000000 2\n,NaN



📊 [problem_solutions.csv] 데이터 상위 5개 행


,problem_id,solution_order,solution,validation
0,2,1,INF = 10000000000.0\nmax_n = 50\nmax_k = 2000\...,NaN
1,2,2,"(n, s, k) = map(int, input().split())\ns -= 1\...",NaN
2,2,3,"import math\n\ndef solve():\n\t(n, s, k) = map...",NaN
3,2,4,"INF = 100000\n(n, s, k) = list(map(int, input(...",NaN
4,2,5,"(n, s, k) = map(int, input().split())\nr = lis...",NaN
